In [284]:
%matplotlib inline

import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, FloatSlider, Output, Label, Dropdown, FloatLogSlider, HTML
from scipy.interpolate import RegularGridInterpolator
from IPython.display import display, clear_output, Audio
import sys
import PathGeneration as pg
from Interpolation import SplineTrajectory3D
from mpl_toolkits.mplot3d import Axes3D

build_dir = "cppSimulation/build"
if build_dir not in sys.path:
    sys.path.insert(0, build_dir)

import propagationmod
import resamplemod

In [ ]:
# audio utils

def db_scale(x, amin=1e-10):
    return 20.0 * np.log10(np.maximum(x, amin))

def lin_scale(x):
    return 10**(x / 20)

def extend_noise(x, target_samples):
    if len(x) < target_samples: 
        X = np.fft.rfft(x)
        X *= np.exp(1j*np.random.rand(len(X))*2*np.pi)
        x = np.tile(np.fft.irfft(X), (target_samples-1)//len(x)+1)
    return x[:target_samples]

def extend_audio(x, target_samples):
    if len(x) >= target_samples:
        return x[: target_samples]
    fade = 2**11
    n = len(x)
    window = np.minimum(np.minimum(np.arange(n)/fade, n/fade-np.arange(n)/fade), 1)
    grain = x * (0.5 - 0.5 * np.cos(np.pi*window))
    y = np.zeros(((target_samples-1)//n+2)*n + 4 * fade)
    for i in range(0, len(y)-n-1, n-fade):
        y[i:i+n] += grain
    return y[fade:fade+target_samples]

def get_audio_from_source(source):
    if isinstance(source, str):
        data, sr = sf.read(source, always_2d=True)
        if data.size == 0:
            raise RuntimeError("Empty or unsupported audio file.")
        audio = data[:, 0]
        return (audio, sr)
    elif isinstance(source, tuple):
        return source
    else:
        raise TypeError("bad scene audio type")
    
def propagate_audio(position_data, mic_positions, audio, sr):
    if isinstance(position_data, tuple):
        pos, vel = position_data
        return propagationmod.simulate_audio_propagation(sr, mic_positions, pos, vel, audio)
    else:
        return propagationmod.static_audio_propagation(sr, mic_positions, position_data, audio)

def rms_normalize(x):
    return x / np.sqrt(np.mean(x * x))

def prepare_audio(audio_source, mic_array, position, total_samples, sr):
    
    audio, sr2 = get_audio_from_source(audio_source)
    
    if sr2 != sr:
        audio = resamplemod.sinc_resample(audio, sr/sr2, 16)
    
    audio = rms_normalize(audio)
    audio = extend_audio(audio, total_samples)
    audio = propagate_audio(position, mic_array, audio, sr)

    return audio

def create_uncorrelated_noise(m, n):
    noise = np.random.randn(m, n)
    for i in range(m):
        noise[i] = rms_normalize(noise[i])
    return noise

def create_diffuse_noise(n, micpos, sr):
    return create_uncorrelated_noise(micpos.shape[1], n)
    # m = micpos.shape[1]
    # noise = np.zeros((m, n), dtype=np.float32)
    # vel = np.zeros((3, n), dtype=np.float32)
    # for _ in range(2):
    #     pos = np.random.randn(3, n)
    #     pos /= np.linalg.norm(pos, axis=0)
    #     noise += propagationmod.simulate_audio_propagation(sr, micpos, pos, vel, np.random.randn(n))
    # for i in range(m):
    #     noise[i] = rms_normalize(noise[i])
    # return noise

In [235]:
# scene utils

class Scene:
    target_position = ()
    interferer_position = []
    interferer_audio = []

def prepare_position(position, t):
    if isinstance(position, tuple):
        pos, t2 = position
        return SplineTrajectory3D(t2, pos).get(t)
    return position

def create_prepared_scene(scene, duration, sr):
    prepared_scene = Scene()
    t = np.linspace(0, duration, duration*sr)
    prepared_scene.target_position = prepare_position(scene.target_position, t)
    prepared_scene.interferer_position = [prepare_position(i, t) for i in scene.interferer_position]
    prepared_scene.interferer_audio = [get_audio_from_source(i) for i in scene.interferer_audio]
    return prepared_scene

def scene_center(duration, sr):
    scene = Scene()
    scene.target_position = np.array([0, 0, 10.0])
    return create_prepared_scene(scene, duration, sr)

def scene_stationary(duration, sr):
    scene = Scene()
    scene.target_position = np.array([2.0, 5.0, 5.0])
    return create_prepared_scene(scene, duration, sr)

def scene_line(duration, sr):
    scene = Scene()
    scene.target_position = pg.straight_line(
        3,
        duration,
        np.array([5, -5, 5]),
        np.array([5, 5, 5]))
    return create_prepared_scene(scene, duration, sr)

def scene_circle(duration, sr):
    scene = Scene()
    scene.target_position = pg.circular_orbit(
        100,
        duration,
        radius=5)
    return create_prepared_scene(scene, duration, sr)

def scene_helix(duration, sr):
    scene = Scene()
    scene.target_position = pg.helix_climb(
        100,
        duration,
        radius=3,
        height=9,
        turns = 2)
    return create_prepared_scene(scene, duration, sr)

def scene_random(duration, sr):
    scene = Scene()
    scene.target_position = pg.random_waypoints_spline(
        6,
        duration,
        np.array([-7, -7, 1]),
        np.array([7, 7, 10])
    )
    return create_prepared_scene(scene, duration, sr)

def get_position(position, t_idx):
    if isinstance(position, tuple):
        return position[0][:, t_idx]
    else:
        return position

In [265]:
# microphone arrays

def square4():
    return np.array(
    [
        [-0.1, -0.1, 0],
        [-0.1,  0.1, 0],
        [ 0.1, -0.1, 0],
        [ 0.1,  0.1, 0],
    ]).T

def square8():
    return np.array(
    [
        [-0.2, -0.2, 0],
        [-0.2,  0.2, 0],
        [ 0.2, -0.2, 0],
        [ 0.2,  0.2, 0],
        [-0.0, -0.2, 0],
        [-0.2,  0.0, 0],
        [ 0.2, -0.0, 0],
        [ 0.0,  0.2, 0],
    ]).T

def random8d01():
    return np.stack((np.random.randn(8), np.random.randn(8), np.zeros(8))) * 0.1

def random16d01():
    return np.stack((np.random.randn(16), np.random.randn(16), np.zeros(16))) * 0.1

In [236]:
# beamforming utils

# half-sphere of look directions
def create_look_directions(n, fov):

    fov += 1e-5
    x = np.linspace(-fov, fov, n)
    y = np.linspace(-fov, fov, n)

    xv, yv = np.meshgrid(x, y)

    # circle the square
    # ratio = np.maximum(np.abs(xv), np.abs(yv)) / (np.sqrt(xv*xv + yv*yv) + 1e-18)
    # xv *= ratio
    # yv *= ratio
    
    aziv = np.atan2(yv, xv)
    zeniv = np.sqrt(xv*xv + yv*yv)

    unit_look_direction = np.stack((
        np.cos(aziv) * np.sin(zeniv),
        np.sin(aziv) * np.sin(zeniv),
        np.cos(zeniv)), axis=-1)
    
    # xy = np.stack((xv, yv), axis = -1)

    return (unit_look_direction, aziv, zeniv, (x, y))

def to_polar_angles(point):
    x, y, z = point
    azi = np.atan2(y, x)
    zeni = np.atan2(np.sqrt(x*x+y*y), z)
    return (azi, zeni)

# look directions (dirs, 3), microphone positions (3, m). returns (dirs, m)
def create_distance_vectors(look_directions, microphone_positions):
    return look_directions @ microphone_positions

def create_steering_vectors(distance_vectors, k):
    return np.exp(1j * distance_vectors * k) / distance_vectors.shape[-1]

# expects shape (samples, mics)
def resonator_filter(audio, halfTime, fc, sr):
    decay = np.pow(0.5, 1 / (halfTime * sr))
    n = audio.shape[0]
    m = audio.shape[1]
    a = np.full(m, np.exp(2j * np.pi * fc / sr) * decay)
    x = np.zeros(audio.shape,  dtype=np.complex64)
    for i in range(1, n):
        x[i] = x[i-1] * a + audio[i-1] * (1.0 - decay)
    return x

# expects shape (mics, samples) return shape (duration, mics)
def get_audio_slice(x, t_idx, duration):
    n = x.shape[1]
    if duration > n:
        return x.T
    s_idx = t_idx
    e_idx = s_idx + duration
    e_idx = min(e_idx, n-1)
    s_idx = e_idx - duration
    return x[:, s_idx:e_idx].T

In [237]:
# Delay and sum

# x (m, w), steering_vectors (dirs, m)
def das_beamform(x, steering_vectors):
    return np.conj(steering_vectors) @ x

def das_beampattern(look_direction, steering_vectors):
    return das_beamform(look_direction, steering_vectors)


In [286]:
# plot utils

def plot_3d_base(ax, azim, elev, size=7.5):
    """
    Plot a 3D flight path and a reference point at the origin.
    points: numpy array of shape (3, N)
    t: numpy array of shape (N,)
    """
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.view_init(elev=elev, azim=azim)

    ax.grid(True)

    ax.set_xlim(-size, size)
    ax.set_ylim(-size, size)
    ax.set_zlim(0, 2 * size)

def plot_path_3d(ax, points, t_idx, color):
    resolution = 2**12
    step = max(1, points.shape[1] // resolution)
    idx = np.arange(0, points.shape[1], step)
    if points.shape[1] % step != 0:
        idx = np.append(idx, [points.shape[1]-1]) 
    p = points[:,idx]
    x, y, z = p

    ax.plot(x, y, z, color=color)
    ax.plot(*points[:,t_idx], color=color, markersize=10, linestyle='', marker='o')

def plot_position_3d(ax, position, t_idx, color):
    if isinstance(position, tuple):
        plot_path_3d(ax, position[0], t_idx, color)
    else:
        ax.plot(*position, color=color, markersize=10, linestyle='', marker='o')

todeg = 1 / np.pi * 180

def plot_mesh_polar(ax, plot_points, z):
    polar_pts, cart_pts, xy = plot_points
    interp = RegularGridInterpolator(xy, np.abs(z), bounds_error=False)
    z = interp(cart_pts)
    c = ax.pcolormesh(*polar_pts, z, shading='auto', cmap='viridis')
    plt.colorbar(c, ax=ax)

In [299]:
c = 344
total_duration = 10
fig_width = 11
dpi = 80
sr = 48000
beamforming_resolution = 100
fov = 90 / todeg

look_directions, aziv, zeniv, xy = create_look_directions(beamforming_resolution, fov)
polar_pts = np.meshgrid(
    np.linspace(0, 2 * np.pi, 7 * beamforming_resolution),
    np.linspace(0, fov * todeg, beamforming_resolution))
cart_pts = np.stack((
    np.sin(polar_pts[0]) * polar_pts[1] / todeg,
    np.cos(polar_pts[0]) * polar_pts[1] / todeg), axis = -1)
plot_points = (polar_pts, cart_pts, xy)

microphone_arrays = {
    "square4": square4,
    "square8": square8,
    "random8d01": random8d01,
    "random16d01": random16d01
}

n = total_duration * sr

audio_sources = {
    "noise": (np.random.randn(n), sr),
    "mavic4":  "../samples/mavic4_1s.wav",
}

scenes = {
    "stationary": scene_stationary,
    "center": scene_center,
    "line": scene_line,
    "circle": scene_circle,
    "helix": scene_helix,
    "random": scene_random
}

class State:
    scene: Scene = None
    microphone_array_pos = np.zeros((3, 1), dtype=np.float32)
    distance_vectors = np.zeros((1, 1), dtype=np.float32)
    target = np.zeros((1, 1), dtype=np.float32)
    noise = np.zeros((1, 1), dtype=np.float32)
    diffuse = np.zeros((1, 1), dtype=np.float32)
    interferer = np.zeros((1, 1), dtype=np.float32)
    audio = np.zeros((1, 1), dtype=np.float32)
    
state = State()

w_array = Dropdown(options=[*microphone_arrays.keys()], value=[*microphone_arrays.keys()][0], description="Array:")
w_target = Dropdown(options=[*audio_sources.keys()], value=[*audio_sources.keys()][0], description="Target:")
w_scene = Dropdown(options=[*scenes.keys()], value=[*scenes.keys()][0], description="Scene:")
w_NFFT = Dropdown(options=[2**i for i in range(6, 14)], value=1024, description="NFFT:")
w_algo = Dropdown(options=["DaS", "MVDR", "MUSIC"], value="DaS", description="Algorithm:")

w_time = FloatSlider(
    value=0.0, min=0.0, max=max(0.0, total_duration - 0.01),
    step=max(0.001, total_duration / 1000.0),
    description="Time (s)", continuous_update=True, readout_format=".2f"
)
w_azi = FloatSlider(
    value=30, min=-180, max=180, step=1,
    description="Azi (deg)", continuous_update=True, readout_format=".0f"
)
w_ele = FloatSlider(
    value=30, min=-90, max=90, step=1,
    description="Ele (deg)", continuous_update=True, readout_format=".0f"
)



w_noise_db = FloatSlider(
    value=-80, min=-80, max=20, step=0.1,
    description="Noise (dB)", continuous_update=True, readout_format=".0f"
)
w_diffuse_db = FloatSlider(
    value=-80, min=-80, max=20, step=0.1,
    description="Diffuse (dB)", continuous_update=True, readout_format=".0f"
)
w_interferer_db = FloatSlider(
    value=-80, min=-80, max=20, step=0.1,
    description="Interf. (dB)", continuous_update=True, readout_format=".0f"
)
w_target_db = FloatSlider(
    value=0, min=-80, max=20, step=0.1,
    description="Target (dB)", continuous_update=True, readout_format=".0f"
)
w_freq = FloatSlider(
    value=1000, min=100, max=4000, step=1,
    description="Freq (Hz)", continuous_update=True, readout_format=".0f"
)
for i in [w_time, w_azi, w_ele, w_noise_db, w_diffuse_db, w_interferer_db, w_target_db, w_freq]:
    i.layout.width = f"{fig_width * dpi * 0.6}px"

# look directions

out1 = Output()
out2 = Output()

def update_plot(change=None):
    time = max(0.0, min(w_time.value, total_duration))
    t_idx = int(round(time * sr))
    t_idx = max(0, min(t_idx, n-1))

    mp = state.microphone_array_pos
    dv = state.distance_vectors

    k = 2 * np.pi * w_freq.value / c
    steering_vectors = create_steering_vectors(dv, k)

    z_view = np.zeros(aziv.shape)
    z_pickup = np.zeros(aziv.shape)

    target_direction = get_position(state.scene.target_position, t_idx)
    unit_target_direction = target_direction / (np.sqrt(np.sum(target_direction*target_direction)) + 1e-18)
    target_polar = np.array(to_polar_angles(target_direction))
    target_polar[1] *= todeg

    x = resonator_filter(get_audio_slice(state.audio, t_idx, int(sr*0.1)), 0.03, w_freq.value, sr)[-1]
    z_view = das_beamform(x, steering_vectors)

    v0 = create_steering_vectors(create_distance_vectors(np.array([unit_target_direction]), mp), k)
    z_pickup = das_beampattern(v0[0], steering_vectors)

    with out1:
        clear_output(wait=True)
        fig = plt.figure(figsize=(fig_width*0.5, fig_width*0.35))
        ax = fig.add_subplot(111, projection='3d')
        plot_3d_base(ax, w_azi.value, w_ele.value)
        ax.plot(0,0,0 , color='red', markersize=4, linestyle='', marker='o')
        ax.plot(*(mp*10) , color='red', markersize=1, linestyle='', marker='o')
        # plt.gca().tick_params(labelbottom=False, labelleft=False)
        
        plot_position_3d(ax, state.scene.target_position, t_idx, 'blue')
        plt.tight_layout()
        plt.show()

    with out2:
        clear_output(wait=True)
        fig = plt.figure(figsize=(fig_width, fig_width*0.4))

        # ax = fig.add_subplot(121)
        # xv, yv = np.meshgrid(*plot_points[2])
        # cc = ax.pcolormesh(xv, yv, np.abs(z_pickup), shading='auto', cmap='viridis')
        # plt.colorbar(cc, ax=ax)

        ax_view = fig.add_subplot(121, projection='polar')
        plot_mesh_polar(ax_view, plot_points, z_view)
        ax_view.scatter(*target_polar, color='blue', s=100)
        ax_view.set_title("Acoustic view")

        ax_pickup = fig.add_subplot(122, projection='polar')
        plot_mesh_polar(ax_pickup, plot_points, z_pickup)
        ax_pickup.scatter(*target_polar, color='blue', s=100)
        ax_pickup.set_title("Pickup pattern")
        
        plt.tight_layout()
        plt.show()

def update_audio(change=None, update_plt=True):
    g_n = lin_scale(w_noise_db.value)
    g_t = lin_scale(w_target_db.value)
    g_i = lin_scale(w_interferer_db.value)
    g_d = lin_scale(w_diffuse_db.value)
    state.audio = state.target * g_t + state.noise * g_n + state.interferer * g_i + state.diffuse * g_d
    if update_plt:
        update_plot()

def update_scene(change=None, update_plt=True):
    scene = scenes[w_scene.value](total_duration, sr)
    state.scene = scene
    m = state.microphone_array_pos.shape[1]

    state.target = prepare_audio(
        audio_sources[w_target.value],
        state.microphone_array_pos,
        scene.target_position, n, sr)
    
    state.interferer = np.zeros((m, n), dtype=np.float32)
    for i in range(len(scene.interferer_audio)):
        state.interferer += prepare_audio(
            scene.interferer_audio[i], 
            state.microphone_array_pos, 
            scene.interferer_position[i], n, sr)
    
    update_audio(update_plt=update_plt)

def update_mic_array(change=None, update_plt=True):
    mic_array = microphone_arrays[w_array.value]()
    state.microphone_array_pos = mic_array
    m = mic_array.shape[1]
    state.noise = create_uncorrelated_noise(m, n)
    state.diffuse = create_diffuse_noise(n, mic_array, sr)
    state.distance_vectors = create_distance_vectors(look_directions, mic_array)
    update_scene(update_plt=update_plt)

w_array.observe(update_mic_array, names="value")

for w in (w_target, w_scene):
    w.observe(update_scene, names="value")

for w in (w_time, w_azi, w_ele, w_NFFT, w_algo):
    w.observe(update_plot, names="value")

for w in (w_target_db, w_noise_db, w_diffuse_db, w_interferer_db, w_freq):
    w.observe(update_audio, names="value")

display(VBox([
    HBox([w_array, w_target, w_scene, w_algo, w_NFFT]),
    HBox([
        VBox([w_time, w_ele, w_azi,
              Label("Noise levels"),
              w_target_db, w_noise_db, w_interferer_db, w_diffuse_db, w_freq
              ]),
        out1
    ]),
    out2
]))

update_mic_array()